<div align="center">

# Week 04: The Human III: Movement, Pointing and Input

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-04) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/Week04_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also defines `normal_cdf` for the normal distribution and three groups of patients of the clinic app with illustrative spreads of their taps. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


def normal_cdf(z):
    """Share of standard normal values below z; works on numbers and arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(z, float) / np.sqrt(2)))


# Three groups of patients of the clinic app (illustrative values): the steady
# spread of their taps sigma0 in mm and the amplitude A of a tremor in mm.
CLINIC_USERS = {"typical adult": (2.0, 0.0), "older adult": (3.0, 0.0),
                "older adult with tremor": (3.0, 4.0)}
print("Setup complete. Running in Colab:", IN_COLAB,
      "| user groups:", len(CLINIC_USERS))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 4.1": (lambda: float(np.polyfit([1.5, 2.5, 3.5, 4.5], [0.35, 0.47, 0.60, 0.70], 1)[0]), 1e-6),
    "Exercise 4.2": (lambda: float(1 - np.sum((np.array([0.35, 0.47, 0.60, 0.70]) - np.polyval(np.polyfit([1.5, 2.5, 3.5, 4.5], [0.35, 0.47, 0.60, 0.70], 1), [1.5, 2.5, 3.5, 4.5])) ** 2) / np.sum((np.array([0.35, 0.47, 0.60, 0.70]) - 0.53) ** 2)), 1e-6),
    "Exercise 4.3": (lambda: float(np.mean(np.abs(ends) <= 3)), 1e-9),
    "Exercise A": (lambda: 0.10 + 0.16 * np.log2(40 / 7 + 1), 1e-6),
    "Exercise B": (lambda: np.log2(80 / (4.133 * 2.0) + 1) / 0.45, 1e-6),
    "Exercise C": (lambda: float(p_hit(15, sigma_total(2.5, 6))), 1e-6),
    "Exercise D": (lambda: 0.20 + 0.06 * 60 / 5, 1e-6),
    "Exercise E": (lambda: float(STAGE4["width for 95 percent, older adult with tremor"]), 1e-6),
}

## Python step 4: Least squares and residuals

Every law of this week is a straight line between a measured time and an index of difficulty. This step fits such a line by hand with the formulas of least squares and checks it against `np.polyfit`. It then measures what the line leaves unexplained and summarises the spread of tap positions with NumPy &#91;[2](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#ref-2)&#93;. The spread becomes the effective width of the lecture &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#ref-1)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "CLINIC_USERS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-04/"
               "NB04_pointing_and_input.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### A line by hand

In [ ]:
# Python practice: a small example for learning the language
ids = np.array([1.81, 2.58, 3.46, 4.39])     # index of difficulty (bits)
mt = np.array([0.42, 0.53, 0.66, 0.79])      # mean movement time (s)
x_bar, y_bar = ids.mean(), mt.mean()
b = np.sum((ids - x_bar) * (mt - y_bar)) / np.sum((ids - x_bar) ** 2)
a = y_bar - b * x_bar
print(f"by hand:    MT = {a:.3f} + {b:.3f} x ID")
b_np, a_np = np.polyfit(ids, mt, 1)
print(f"np.polyfit: MT = {a_np:.3f} + {b_np:.3f} x ID")

The *least-squares* line makes the sum of the squared vertical distances to the points as small as possible. Its slope is $b = \sum (x_i - \bar{x})(y_i - \bar{y}) / \sum (x_i - \bar{x})^2$ and its intercept $a = \bar{y} - b\bar{x}$, where a bar marks a mean. Arithmetic on whole arrays computes all terms at once, and `np.sum` adds them. `np.polyfit` returns the same line, which confirms the formulas.

### Residuals and the coefficient of determination

In [ ]:
# Python practice: a small example for learning the language
pred = a + b * ids
resid = mt - pred
print("residuals (s):", resid.round(4))
ss_res = np.sum(resid ** 2)
ss_tot = np.sum((mt - y_bar) ** 2)
r2 = 1 - ss_res / ss_tot
print(f"R^2 = {r2:.4f}")

A *residual* is the difference between a measured value and the value of the line. Small residuals without a pattern mean that the line describes the data. The *coefficient of determination* $R^2 = 1 - \sum r_i^2 / \sum (y_i - \bar{y})^2$ is the share of the variation of the times that the line explains. A value of 1 means a perfect fit, and 0 means that the line does no better than the mean.

### The spread of tap positions

In [ ]:
# Python practice: a small example for learning the language
rng = np.random.default_rng(4)
ends = rng.normal(loc=0.0, scale=2.5, size=500)   # where 500 taps land (mm)
sd = ends.std(ddof=1)
print(f"standard deviation {sd:.2f} mm")
print(f"share within 4.5 mm of the centre: {np.mean(np.abs(ends) <= 4.5):.3f}")
print(f"normal model: {float(2 * normal_cdf(4.5 / sd) - 1):.3f}")

`rng.normal(loc, scale, size)` draws values from a normal distribution with the mean `loc` and the standard deviation `scale`. The method `std(ddof=1)` divides by $n - 1$, the usual estimate from a sample. The share of taps near the centre can be counted directly or computed from the normal model with `normal_cdf` of the setup cell, and the two agree closely. A distance of 4.5 mm on each side matches a target 9 mm wide.

### Plotting a fit with its residuals

In [ ]:
# Python practice: a small example for learning the language
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
xs = np.linspace(0, 5, 50)
ax[0].plot(ids, mt, "o", color="#C98510", label="measured means")
ax[0].plot(xs, a + b * xs, color="#5A4BB5", label="least-squares line")
ax[0].set_xlabel("index of difficulty (bits)")
ax[0].set_ylabel("movement time (s)")
ax[0].legend()
ax[1].axhline(0, color="#585E6E", lw=1)
ax[1].stem(ids, resid)
ax[1].set_xlabel("index of difficulty (bits)")
ax[1].set_ylabel("residual (s)")
plt.tight_layout()
plt.show()

The right panel shows each residual as a stem from zero. Residuals that rise and fall without a trend support the straight line. Residuals that bend in a curve would call for another model.

**Quick check.** Why can $R^2$ never exceed 1?

<details><summary>Answer</summary>

The sum of squared residuals cannot be negative, so 1 minus a ratio that is zero or positive is at most 1. The value 1 is reached only when every point lies on the line.

</details>

### Exercises for Python step 4

**Exercise 4.1.** Times of 0.35, 0.47, 0.60 and 0.70 s were measured at 1.5, 2.5, 3.5 and 4.5 bits. Compute the least-squares slope with the formula, not with `np.polyfit`, and store it in `b_hand`.

In [ ]:
b_hand = None   # your answer

check("Exercise 4.1", b_hand)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
x = np.array([1.5, 2.5, 3.5, 4.5])
y = np.array([0.35, 0.47, 0.60, 0.70])
b_hand = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)

check("Exercise 4.1", b_hand)

**Exercise 4.2.** Store the coefficient of determination $R^2$ of the same fit in `r2_hand`.

In [ ]:
r2_hand = None   # your answer

check("Exercise 4.2", r2_hand)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
x = np.array([1.5, 2.5, 3.5, 4.5])
y = np.array([0.35, 0.47, 0.60, 0.70])
slope, icpt = np.polyfit(x, y, 1)
res = y - (icpt + slope * x)
r2_hand = 1 - np.sum(res ** 2) / np.sum((y - y.mean()) ** 2)

check("Exercise 4.2", r2_hand)

**Exercise 4.3.** Store the share of the 500 simulated taps `ends` that land within 3 mm of the centre in `share_6`. It is the hit rate of a target 6 mm wide.

In [ ]:
share_6 = None   # your answer

check("Exercise 4.3", share_6)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
share_6 = np.mean(np.abs(ends) <= 3)

check("Exercise 4.3", share_6)

## 1. Fitts' law: Your ring of taps &#91;[4](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#ref-4)&#93;

*Lecture: [Fitts' law](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#fitts-law)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "ends" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-04/"
               "NB04_pointing_and_input.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A ring of taps produces dozens of movement times for six combinations of distance and width. Python turns each combination into an index of difficulty, averages the times and fits the two constants of the law. A table of raw times would hide all three.

**From the formulas to the code.** The distance $D$ is `D`, the width $W$ is `W` and the index of difficulty (ID) $\log_2(D/W + 1)$ is `fitts_id(D, W)`. The function `fitts_time` returns the movement time (MT) $a + b \cdot ID$, with $a$ as `a` and $b$ as `b`.

In [ ]:
def fitts_id(D, W):
    """Index of difficulty in bits, in the Shannon form."""
    return np.log2(np.asarray(D, float) / np.asarray(W, float) + 1)

def fitts_time(D, W, a=0.10, b=0.16):
    """Predicted movement time in s."""
    return a + b * fitts_id(D, W)

ID_ex, MT_ex = float(fitts_id(160, 20)), float(fitts_time(160, 20))
print(f"worked example, D = 160 mm and W = 20 mm: {ID_ex:.3f} bits, {MT_ex:.3f} s")
print("half the width, W = 10 mm:", round(float(fitts_time(160, 10)), 3), "s")

**Python note.** `np.asarray(..., float)` lets the functions accept single numbers and whole arrays alike, so the same code computes one prediction or a whole table of them.

**Exercise A.** A time chip of the clinic app is 7 mm wide and 40 mm from the thumb. Store its predicted movement time with $a = 0.10$ s and $b = 0.16$ s per bit in `mt_chip`, with the formula and without calling `fitts_time`.

In [ ]:
mt_chip = None   # your answer

check("Exercise A", mt_chip)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
mt_chip = 0.10 + 0.16 * np.log2(40 / 7 + 1)

check("Exercise A", mt_chip)

Part A of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text between the triple quotes of `MY_TAPS`. Left empty, the variable makes the code use a simulated class of 24 students. Their times and spreads are assumptions of this Colab notebook.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_TAPS = """
"""

def simulated_tap_class(n_students=24, seed=RANDOM_STATE):
    """Nine timed taps for each of six combinations of distance and width."""
    rng = np.random.default_rng(seed)
    rows = []
    for student in range(1, n_students + 1):
        a, b = rng.normal(0.12, 0.03), rng.normal(0.17, 0.03)
        for D in (80, 160):
            for W in (8, 16, 32):
                sd = W / 4.133 * rng.uniform(0.8, 1.2)   # spread near the target
                for trial in range(1, 10):
                    dx, dy = rng.normal(0, sd), rng.normal(0, sd * 0.5)
                    mt = (a + b * fitts_id(D, W)) * rng.lognormal(0, 0.12)
                    hit = int(np.hypot(dx, dy) <= W / 2)
                    rows.append([student, D, W, trial, D, round(dx, 2), round(dy, 2),
                                 hit, round(1000 * mt)])
    return pd.DataFrame(rows, columns=["student", "D_mm", "W_mm", "trial", "dist_mm",
                                       "dx_mm", "dy_mm", "hit", "mt_ms"])

if MY_TAPS.strip():
    taps = pd.read_csv(io.StringIO(MY_TAPS.strip()))
    taps["student"] = 1
    source = "your own taps from the interactive lab"
else:
    taps = simulated_tap_class()
    source = "a simulated class of 24 students"
print("Data:", source, "|", len(taps), "taps")
cond = taps.groupby(["D_mm", "W_mm"]).agg(mt_s=("mt_ms", "mean"),
                                         hits=("hit", "mean"))
cond["mt_s"] /= 1000
cond["ID"] = fitts_id(cond.index.get_level_values("D_mm"),
                      cond.index.get_level_values("W_mm"))
print(cond.round(3))

**Python note.** `agg` computes several summaries per group at once and names the new columns. `get_level_values` reads one level of the two-level index, here the distances and the widths, so that the index of difficulty can be computed for every row.

In [ ]:
b_fit, a_fit = np.polyfit(cond["ID"], cond["mt_s"], 1)
pred = a_fit + b_fit * cond["ID"]
ss_res = np.sum((cond["mt_s"] - pred) ** 2)
r2 = 1 - ss_res / np.sum((cond["mt_s"] - cond["mt_s"].mean()) ** 2)
fit_ab = (a_fit, b_fit)
print(f"MT = {a_fit:.3f} + {b_fit:.3f} x ID   (R^2 = {r2:.3f})")
print(f"each additional bit costs {1000 * b_fit:.0f} ms")

In [ ]:
# Figure: movement time against the index of difficulty, with the fitted line
fig, ax = plt.subplots(figsize=(7, 3.6))
x = np.linspace(1, 5, 50)
for D, marker in [(80, "o"), (160, "s")]:
    d = cond.xs(D, level="D_mm")
    ax.plot(d["ID"], d["mt_s"], marker, color="#C98510", label=f"ring D = {D} mm")
ax.plot(x, a_fit + b_fit * x, color="#5A4BB5",
        label=f"MT = {a_fit:.2f} + {b_fit:.3f} ID")
ax.set_xlabel("index of difficulty ID (bits)")
ax.set_ylabel("mean movement time (s)")
ax.legend()
plt.tight_layout()
plt.show()

**Python note.** `cond.xs(D, level="D_mm")` selects the rows of one distance from a table with a two-level index. Different markers for the two distances show whether both fall on one line, as the law predicts.

**Observe and reflect.** Do the points of the two distances fall on one line, or does the longer distance cost more than its index of difficulty says? What would that mean for the large screens of kiosks?

## 2. Effective width and throughput &#91;[6](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#ref-6)&#93;

*Lecture: [Effective width and throughput](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#throughput)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "fit_ab" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-04/"
               "NB04_pointing_and_input.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** People do not aim at the edge of a target, and each person trades speed for accuracy in a personal way. The spread of the endpoints shows how accurate the taps really were. Python turns it into an effective width and a throughput, which can be compared across people and devices.

**From the formulas to the code.** The standard deviation $\sigma$ of the endpoints along the direction of movement is `sd`, and the effective width $W_e = 4.133\,\sigma$ is `We`. The effective distance $D_e$ is `De`, the effective index of difficulty $\log_2(D_e/W_e + 1)$ is `IDe` and the throughput (TP) $TP = ID_e / MT$ is `TP`.

In [ ]:
sd, De, MT = 3.2, 160, 0.64            # the worked example of the lecture
We = 4.133 * sd
IDe = np.log2(De / We + 1)
print(f"worked example: We = {We:.2f} mm, IDe = {IDe:.2f} bits,",
      f"TP = {IDe / MT:.2f} bits/s")

def throughput(dist, dx, mt_s):
    """Effective width (mm), effective ID (bits) and throughput (bits/s)."""
    sd = np.std(dx, ddof=1)
    We = 4.133 * sd
    De = np.mean(np.asarray(dist, float) + np.asarray(dx, float))
    IDe = np.log2(De / We + 1)
    return We, IDe, IDe / np.mean(mt_s)

**Python note.** A function may return several values at once. Python packs them into a tuple, and the caller unpacks them into names, as in `We, IDe, TP = throughput(...)` in the next cell.

**Exercise B.** A participant aimed at a target 80 mm away, with endpoints spread by 2.0 mm and a mean time of 0.45 s. Store the throughput in bits per second in `tp_b`, with $D_e = 80$ mm.

In [ ]:
tp_b = None   # your answer

check("Exercise B", tp_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
tp_b = np.log2(80 / (4.133 * 2.0) + 1) / 0.45

check("Exercise B", tp_b)

In [ ]:
rows = []
for (student, D, W), d in taps.groupby(["student", "D_mm", "W_mm"]):
    We, IDe, TP = throughput(d["dist_mm"], d["dx_mm"], d["mt_ms"] / 1000)
    rows.append([student, D, W, float(fitts_id(D, W)), We, IDe, TP])
eff = pd.DataFrame(rows, columns=["student", "D_mm", "W_mm", "ID", "We_mm",
                                 "IDe", "TP"])
per_student = eff.groupby("student")["TP"].mean()
tp_mean = per_student.mean()
print(eff.groupby(["D_mm", "W_mm"])[["ID", "We_mm", "IDe", "TP"]].mean().round(2))
print(f"throughput, mean of the means: {tp_mean:.2f} bits/s "
      f"(students from {per_student.min():.2f} to {per_student.max():.2f})")

**Python note.** The throughput is computed for each condition of each student, averaged per student and then over the students. This *mean of means* follows the recommendation of Soukoreff and MacKenzie, and a single fast or slow condition cannot dominate it.

In [ ]:
m = eff.groupby(["D_mm", "W_mm"])[["We_mm"]].mean()
labels = [f"D {D}, W {W}" for D, W in m.index]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].bar(labels, m["We_mm"], color="#5A4BB5", label="effective width")
ax[0].plot(labels, [W for D, W in m.index], "o", color="#C98510",
           label="nominal width")
ax[0].set_ylabel("width (mm)")
ax[0].tick_params(axis="x", rotation=30)
ax[0].legend()
ax[1].hist(per_student, bins=10, color="#0E7A78")
ax[1].set_xlabel("throughput per student (bits/s)")
ax[1].set_ylabel("students")
plt.tight_layout()
plt.show()

**Python note.** `tick_params(axis="x", rotation=30)` turns the labels of the bars, so that long labels do not overlap. A histogram with `bins=10` shows how the throughput spreads across the students.

**Observe and reflect.** Is the effective width larger or smaller than the nominal width for the small targets? What does each case say about how the taps traded speed for accuracy?

## 3. Touch and tremor: The probability of a hit &#91;[7](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#ref-7)&#93;

*Lecture: [Touch, fingers and tremor](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#touch-and-tremor)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "tp_mean" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-04/"
               "NB04_pointing_and_input.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Whether a tap hits a button depends on the spread of the taps and on the size of the button. Python combines the steady spread with a tremor, computes the probability of a hit for any size and finds the size that a group of patients needs. No single test session could show all of that.

**From the formulas to the code.** The steady spread $\sigma_0$ is `s0`, the amplitude of the tremor $A$ is `A`, and the combined spread $\sigma = \sqrt{\sigma_0^2 + A^2/2}$ is `sigma_total(s0, A)`. The probability of a hit on a square button of width $W$, $P = (2\Phi(W/2\sigma) - 1)^2$, is `p_hit(W, sigma)`, with $\Phi$ as `normal_cdf`.

In [ ]:
def sigma_total(s0, A):
    """Spread of taps (mm): steady part s0, sinusoidal tremor of amplitude A."""
    return np.sqrt(np.asarray(s0, float) ** 2 + np.asarray(A, float) ** 2 / 2)

def p_hit(W, sigma, square=True):
    """Probability that a tap lands on a target of width W (mm)."""
    p1 = 2 * normal_cdf(np.asarray(W, float) / (2 * np.asarray(sigma, float))) - 1
    return p1 ** 2 if square else p1

def width_for(p, sigma):
    """Width (mm) of a square button with hit probability p, by bisection."""
    lo, hi = 0.0, 100.0
    for _ in range(50):
        mid = (lo + hi) / 2
        if p_hit(mid, sigma) < p:
            lo = mid
        else:
            hi = mid
    return hi

s = float(sigma_total(3.0, 4.0))
print(f"worked example: sigma = {s:.2f} mm")
print(f"9 mm button: typical adult {float(p_hit(9, 2.0)):.3f},",
      f"with tremor {float(p_hit(9, s)):.3f}")
print(f"width for 95 percent: {width_for(0.95, 2.0):.1f} mm and",
      f"{width_for(0.95, s):.1f} mm")

**Python note.** `width_for` uses *bisection*: It keeps an interval that contains the answer and halves it fifty times, always keeping the half where the probability crosses `p`. The lecture rounds every intermediate value, so its last digit can differ by one from the exact values printed here.

**Exercise C.** A patient with a steady spread of 2.5 mm and a tremor of amplitude 6 mm taps a square button 15 mm wide. Store the probability of a hit in `p_c`.

In [ ]:
p_c = None   # your answer

check("Exercise C", p_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_c = p_hit(15, sigma_total(2.5, 6))

check("Exercise C", p_c)

Part C of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_TREMOR`. Left empty, the variable makes the code use a simulated class of 24 students with assumed steady spreads.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_TREMOR = """
"""

def simulated_tremor_class(n_students=24, seed=RANDOM_STATE):
    """Four taps on each of five widths, steady and with a tremor of 4 mm."""
    rng = np.random.default_rng(seed + 1)
    rows = []
    for student in range(1, n_students + 1):
        s0 = rng.uniform(1.2, 2.4)
        for block, A in [(1, 0), (2, 4)]:
            s = float(sigma_total(s0, A))
            for W in (6, 9, 12, 16, 20):
                for trial in range(1, 5):
                    dx, dy = rng.normal(0, s), rng.normal(0, s)
                    hit = int(abs(dx) <= W / 2 and abs(dy) <= W / 2)
                    mt = 0.45 + 0.16 * np.log2(100 / W + 1)
                    mt *= rng.lognormal(0, 0.15)
                    rows.append([student, block, A, W, trial, round(dx, 2),
                                 round(dy, 2), hit, round(1000 * mt)])
    cols = ["student", "block", "tremor_mm", "W_mm", "trial", "dx_mm", "dy_mm",
            "hit", "mt_ms"]
    return pd.DataFrame(rows, columns=cols)

if MY_TREMOR.strip():
    tremor = pd.read_csv(io.StringIO(MY_TREMOR.strip()))
    tremor["student"] = 1
    source = "your own taps from the interactive lab"
else:
    tremor = simulated_tremor_class()
    source = "a simulated class of 24 students"
print("Data:", source, "|", len(tremor), "taps")
sigma_hat = {}
for A, d in tremor.groupby("tremor_mm"):
    var = (np.var(d["dx_mm"], ddof=1) + np.var(d["dy_mm"], ddof=1)) / 2
    sigma_hat[A] = float(np.sqrt(var))
s0_hat = sigma_hat[min(sigma_hat)]
for A, s in sigma_hat.items():
    print(f"tremor {A} mm: measured spread {s:.2f} mm, "
          f"model {float(sigma_total(s0_hat, A)):.2f} mm")

**Python note.** The spread is the root of the mean of the two variances, across and along the screen. The model starts from the steady spread of the first block and adds the tremor, so the second line tests the formula of the lecture on new taps.

In [ ]:
# Figure: probability of a hit against the width of the button
obs = tremor.groupby(["tremor_mm", "W_mm"])["hit"].mean()
w = np.linspace(2, 24, 100)
fig, ax = plt.subplots(figsize=(7, 3.6))
for A, col in zip(sorted(sigma_hat), ["#5A4BB5", "#B53F2E"]):
    ax.plot(w, p_hit(w, sigma_hat[A]), color=col, label=f"model, tremor {A} mm")
    o = obs.xs(A, level="tremor_mm")
    ax.plot(o.index, o.values, "o", color=col, label=f"observed, tremor {A} mm")
    print(f"tremor {A} mm: 95 percent needs {width_for(0.95, sigma_hat[A]):.1f} mm")
ax.axhline(0.95, ls=":", color="#585E6E")
ax.set_xlabel("width of the square button W (mm)")
ax.set_ylabel("probability of a hit")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Python note.** `zip` walks through two lists in step, here the tremor levels and their colours. The dotted line marks a hit rate of 95 percent, where `width_for` reads the required width.

**Observe and reflect.** At which width does each curve reach 0.95? How many times wider must a button be for a patient with tremor than for a patient without?

## 4. Steering: Your tunnels &#91;[10](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#ref-10)&#93;

*Lecture: [The steering law](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#steering-law)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "sigma_hat" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-04/"
               "NB04_pointing_and_input.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Steering through a tunnel is a different task from tapping, because the whole path must stay inside. Python fits the steering law to your runs and compares it with Fitts' law for the same sizes. The comparison shows how much more a narrow path costs than a small target.

**From the formulas to the code.** The length of the tunnel $A$ is `A`, its width $W$ is `W` and the index of difficulty of steering $A/W$ is `A / W`. The function `steer_time` returns $a + b\,A/W$.

In [ ]:
def steer_time(A, W, a=0.20, b=0.06):
    """Predicted time (s) to steer through a straight tunnel A long, W wide."""
    return a + b * np.asarray(A, float) / np.asarray(W, float)

print("worked example, slide to confirm, A = 120 mm and W = 12 mm:",
      round(float(steer_time(120, 12)), 3), "s")
print("a track twice as wide:", round(float(steer_time(120, 24)), 3), "s")

**Exercise D.** A cascading menu requires a horizontal move of 60 mm inside an item 5 mm high. Store the steering time with $a = 0.20$ s and $b = 0.06$ s in `t_d`.

In [ ]:
t_d = None   # your answer

check("Exercise D", t_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_d = 0.20 + 0.06 * 60 / 5

check("Exercise D", t_d)

Part D of the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lab.html) ends with the button *Copy the results for Colab*. Paste the copied text into `MY_TUNNELS`. Left empty, the variable makes the code use a simulated class of 24 students with assumed constants.

In [ ]:
# Paste the results of lab Part D between the triple quotes.
MY_TUNNELS = """
"""

def simulated_tunnel_class(n_students=24, seed=RANDOM_STATE):
    """Two runs through each of six straight tunnels."""
    rng = np.random.default_rng(seed + 2)
    rows = []
    for student in range(1, n_students + 1):
        a, b = rng.normal(0.25, 0.05), rng.normal(0.055, 0.012)
        for A in (120, 240):
            for W in (10, 20, 40):
                for trial in (1, 2):
                    t = (a + b * A / W) * rng.lognormal(0, 0.12)
                    restarts = rng.poisson(0.08 * A / W)
                    rows.append([student, f"{A}x{W}", A, W, trial, round(1000 * t),
                                 restarts])
    return pd.DataFrame(rows, columns=["student", "tunnel", "A_mm", "W_mm", "trial",
                                       "time_ms", "restarts"])

if MY_TUNNELS.strip():
    tunnels = pd.read_csv(io.StringIO(MY_TUNNELS.strip()))
    tunnels["student"] = 1
    source = "your own runs from the interactive lab"
else:
    tunnels = simulated_tunnel_class()
    source = "a simulated class of 24 students"
print("Data:", source, "|", len(tunnels), "runs")
steer = tunnels.groupby(["A_mm", "W_mm"]).agg(t_s=("time_ms", "mean"),
                                              restarts=("restarts", "mean"))
steer["t_s"] /= 1000
steer["AW"] = (steer.index.get_level_values("A_mm")
               / steer.index.get_level_values("W_mm"))
b_s, a_s = np.polyfit(steer["AW"], steer["t_s"], 1)
steer_ab = (a_s, b_s)
print(steer.round(3))
print(f"T = {a_s:.3f} + {b_s:.4f} x A/W")

**Python note.** The restarts count how often a run left the tunnel and began again. They are kept apart from the times, because the steering law predicts the time of a successful run.

**Explore.** Move the sliders: The panel draws a tunnel of the chosen length and width, with your fitted constants. It compares the steering time with the time to tap a target of the same width at the same distance. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(A=120, W=12):
    t_steer = float(steer_time(A, W, *steer_ab))
    t_tap = float(fitts_time(A, W))
    fig, ax = plt.subplots(1, 2, figsize=(10, 2.4),
                           gridspec_kw={"width_ratios": [2, 1]})
    ax[0].add_patch(plt.Rectangle((0, -W / 2), A, W, color="#C9C3E6"))
    ax[0].plot([0, A], [0, 0], color="#5A4BB5", lw=2)
    ax[0].set_xlim(-10, 250)
    ax[0].set_ylim(-30, 30)
    ax[0].set_aspect("equal")
    ax[0].set_title(f"tunnel {A} mm long and {W} mm wide")
    ax[1].barh(["steer", "tap"], [t_steer, t_tap], color=["#B53F2E", "#5A4BB5"])
    ax[1].set_xlabel("time (s)")
    plt.tight_layout()
    plt.show()
    print(f"steering {t_steer:.2f} s | tapping a target of the same width",
          f"{t_tap:.2f} s")

slider_A = widgets.IntSlider(value=120, min=40, max=240, step=10,
                             description="A (mm)")
slider_W = widgets.IntSlider(value=12, min=4, max=40, step=1, description="W (mm)")
widgets.interact(explore, A=slider_A, W=slider_W);

**Python note.** `*steer_ab` unpacks the fitted intercept and slope into the arguments `a` and `b` of `steer_time`. The library ipywidgets turns the arguments of `explore` into the two sliders defined above.

**Observe and reflect.** For a width of 10 mm, how much longer does steering through 120 mm take than tapping a target 120 mm away? Which controls of an app force users to steer, and which could be replaced by taps?

## 5. The clinic app, stage 4

*Lecture: [The clinic app, stage 4: Targets that fit fingers, tremor and the kiosk](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#the-clinic-app-stage-4)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "steer_ab" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-04/"
               "NB04_pointing_and_input.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The stage answers three questions for the screens of the clinic app. How often does each group of patients hit a time chip, how wide must the chips be, and should the kiosk confirm with a slide or with a button? A dictionary keeps every result of the stage, and Week 9 tests the same designs.

In [ ]:
STAGE4 = {}
for user, (s0, A) in CLINIC_USERS.items():
    s = float(sigma_total(s0, A))
    for W in (7, 12):
        STAGE4[f"hit {W} mm, {user}"] = round(float(p_hit(W, s)), 3)
    STAGE4[f"width for 95 percent, {user}"] = round(width_for(0.95, s), 1)
STAGE4["chip 7 mm, time (s)"] = round(float(fitts_time(40, 7)), 3)
STAGE4["chip 12 mm, time (s)"] = round(float(fitts_time(40, 12)), 3)
STAGE4["slide 120 x 12 mm, time (s)"] = round(float(steer_time(120, 12)), 3)
STAGE4["button 25 mm, time (s)"] = round(float(fitts_time(160, 25)), 3)
tremor_sigma = float(sigma_total(3.0, 4.0))
STAGE4["button 25 mm, hit with tremor"] = round(float(p_hit(25, tremor_sigma)), 3)
for k, v in STAGE4.items():
    print(f"{k:<42} {v}")

**Python note.** Each user group is unpacked into its two values with `for user, (s0, A) in CLINIC_USERS.items()`. The parentheses match the pair stored for each name.

In [ ]:
users = list(CLINIC_USERS)
xpos = np.arange(len(users))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
hits_7 = [STAGE4[f"hit 7 mm, {u}"] for u in users]
hits_12 = [STAGE4[f"hit 12 mm, {u}"] for u in users]
ax[0].bar(xpos - 0.2, hits_7, 0.4, color="#B53F2E", label="chip 7 mm (current)")
ax[0].bar(xpos + 0.2, hits_12, 0.4, color="#2E8B57", label="chip 12 mm")
ax[0].set_xticks(xpos, users)
ax[0].set_ylabel("probability of a hit")
ax[0].set_title("Hitting a time chip")
ax[0].legend(loc="lower left")
widths = [STAGE4[f"width for 95 percent, {u}"] for u in users]
ax[1].bar(users, widths, color="#5A4BB5")
for W, ls in [(7, ":"), (12, "--")]:
    ax[1].axhline(W, ls=ls, color="#585E6E", label=f"chip {W} mm")
ax[1].set_ylabel("width for 95 percent hits (mm)")
ax[1].set_title("Width that each group needs")
ax[1].legend()
plt.tight_layout()
plt.show()

**Python note.** Two sets of bars side by side are drawn by shifting the positions by 0.2 to each side and making each bar 0.4 wide. The horizontal lines mark the current and the proposed width of the chips.

**Exercise E.** Store the width that a patient with tremor needs for a hit rate of 95 percent, as kept in `STAGE4`, in `w_tremor`.

In [ ]:
w_tremor = None   # your answer

check("Exercise E", w_tremor)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
w_tremor = STAGE4["width for 95 percent, older adult with tremor"]

check("Exercise E", w_tremor)

**Observe and reflect.** Chips of 12 mm still miss about one tap in four for a patient with tremor. What else could the app change, besides the size, so that a miss does not book the wrong time?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE4" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-04/"
               "NB04_pointing_and_input.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Each app below has a small target that its users must hit. The functions of Sections 1 and 3 compute the time and the hit rate of that target for the three groups of patients after one change of a name.

The targets below are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "banking"
# choose: clinic, banking, e_government, food_delivery, course_registration
APPS = {"clinic": ("time chip", 7, 40), "banking": ("key of the PIN pad", 10, 30),
        "e_government": ("link in a paragraph", 4, 60),
        "food_delivery": ("plus and minus buttons", 6, 25),
        "course_registration": ("checkbox of a course", 5, 80)}

name, W_app, D_app = APPS[APP]
print(f"{APP}: {name}, {W_app} mm wide and {D_app} mm away, "
      f"MT = {float(fitts_time(D_app, W_app)):.2f} s")
for user, (s0, A) in CLINIC_USERS.items():
    s = float(sigma_total(s0, A))
    print(f"  {user:<24} hit {float(p_hit(W_app, s)):.2f} | "
          f"95 percent needs {width_for(0.95, s):.1f} mm")

**Python note.** `name, W_app, D_app = APPS[APP]` gives the three values of the chosen app their own names in one line. The format `{user:<24}` pads the name to 24 characters, so that the columns line up.

**Observe and reflect.** Which app has the target that fails older users most often? Propose a design that makes the target larger or removes it.

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | A code editor shows breakpoints as dots 3 mm wide in the margin. Compute the time and the hit rate for a developer with a steady spread of 1.5 mm, and propose a size. |
| Web development | A navigation menu opens its submenus on hovering. Model the move into a submenu as a steering task and compare it with a menu that opens on a click. |
| Mobile development | A bottom navigation bar has five icons 10 mm wide. Compute the hit rate for the three groups of patients and the width that the tremor group needs. |
| Game development | A game asks players to tap moving targets. Simulate 100 taps with `rng.normal` and a spread of 3 mm, and estimate the effective width and the throughput. |
| Embedded systems and the Internet of Things | A smart thermostat has a touch ring of 40 mm. Model the slide along a quarter of the ring as a steering task with a track of 8 mm. |
| Data science | Fit Fitts' law for each simulated student of Section 1 separately and report the mean and the spread of the slopes $b$. |
| Cybersecurity | A PIN pad shuffles its keys after each use, so users must search before they tap. Add a search time to Fitts' law and estimate the cost of the shuffling for a 4-digit PIN. |
| Health informatics | An infusion pump has buttons 8 mm wide. Compute the hit rate for a nurse wearing gloves with a steady spread of 3.5 mm, and the width for 99 percent. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-04/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-04).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>